Load and Train Models On the Iris Dataset

In [3]:
#Import
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pickle

#Load
iris = load_iris()
X, y = iris.data, iris.target

# Split and train Random Forest
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# Save the model 
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("DONE")


DONE


API for Model Predictions

In [13]:
from flask import Flask, request, jsonify
app = Flask(__name__)

#Load the model
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

#Define prediction route
@app.route('/predict', methods=['GET'])
def predict():
    feature_1 = float(request.args.get('sepal_length'))
    feature_2 = float(request.args.get('sepal_width'))
    feature_3 = float(request.args.get('petal_length'))
    feature_4 = float(request.args.get('petal_width'))
    prediction = model.predict([[feature_1, feature_2, feature_3, feature_4]])
    confidence = model.predict_proba([[feature_1, feature_2, feature_3, feature_4]]).max()
    return jsonify({
        'model': 'Random Forest',
        'prediction': int([prediction][0]),
        'confidence': confidence}
        )

app.run(port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Jan/2025 18:02:35] "GET /predict?sepal_length=5.1&sepal_width=3.5&petal_length=1.4&petal_width=0.2 HTTP/1.1" 200 -


Metamodel 

In [3]:
import requests
import numpy as np
from collections import Counter
from proof_of_stake import *

# List of API URLs for group members
api_urls = ["https://5fc9-89-30-29-68.ngrok-free.app/predict", "https://50da-89-30-29-68.ngrok-free.app/predict", "https://b05d-89-30-29-68.ngrok-free.app/predict"]

model_weights = {"model1": 0.8, "model2": 0.9, "model3": 0.7}

def adjust_weights(model_name, isCorrect):
    # Increase weight if accurate, decrease otherwise
    model_weights[model_name] = max(0, min(1, model_weights[model_name] + (0.1 if isCorrect else -0.1)))

def get_predictions(features):
    predictions = []
    for url in api_urls:
        response = requests.get(url, params=features)
        if response.status_code == 200:
            predictions.append(response.json()['prediction'])
    # Perform majority voting (choose the most common class)
    if predictions:
        majority_vote = Counter(predictions).most_common(1)[0][0]
        for i in range(3):
            model_name = "model" + str(i)
            if predictions[i]!=majority_vote:
                slash_balance(model_name,10)
                adjust_weights(model_name,False)
            else:
                reward_balance(model_name,10)
                adjust_weights(model_name,True)             
        return majority_vote
    else:
        raise ValueError("No predictions were collected from the APIs.")

# Feature inputs
features = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}

# Get the consensus prediction
consensus_prediction = get_predictions(features)
print("Consensus Prediction:", consensus_prediction)

ValueError: No predictions were collected from the APIs.